In [4]:
from src.server.models.billing.billing_data import BillingUtils

# org user

email = "shum@agnitiocapital.com"
name = "Shum Singh"
company = "THQ Nordic Official"
company_id = 35590
startup = False
modules = [
    BillingUtils.MODULE_AUDITOR,
    BillingUtils.MODULE_TUNER,
    BillingUtils.MODULE_GAPS,
    BillingUtils.MODULE_GAMES_EXPLORER,
    BillingUtils.MODULE_PLAYERS_EXPLORER,
    BillingUtils.MODULE_STUDIES
]

In [5]:
from src.runtime_constants import rr
from src.db.mysql import DbConnection

#conn = DbConnection()
conn = DbConnection(rr.ENV_PROD, is_remote=True)

In [6]:
from src.server.models.user_constants import uc
from src.utils.gembase_utils import GembaseUtils
from src.server.models.organization.organization_model import OrganizationModel
from src.server.models.billing.billing_model import BillingModel
from src.server.models.app_model import AppModel

admin_id = conn.select_one("""
SELECT u.id
  FROM app.users u
 WHERE u.guid = %s
""", [uc.ADMIN_USER_GUID])["id"]
if company_id is None:
    dev_data = AppModel.create_dev_concept(
        conn=conn,
        title=company
    )
    company_id = dev_data["dev_id_int"]
prime_number = conn.select_one("""
SELECT p.id
  FROM app.def_prime_numbers p
  ORDER BY RAND()
  LIMIT 1
""")["id"]
org_id = OrganizationModel.s__create_organization(
    conn=conn,
    dev_id_int=company_id,
    prime_number=prime_number,
    users=[],
    email_domain=None,
    credits_value=25
)
modules_seats = [{"id": m, "seats": 1} for m in modules]
billing_id = BillingModel.s__create(
    conn=conn,
    organization_id=org_id,
    user_id=0,
    payment_id=0,
    expire_months=12,
    modules=modules_seats
)
email_id = UserHelper.get_or_create_email_id(conn=conn, email=email)
request_id = conn.insert("""
insert into app.organization_requests (organization_id, email, request_guid, from_user_id, role, name, locked)
VALUES (%s, %s, %s, %s, 'member', %s, 1)
""", [org_id, email, GembaseUtils.get_guid(), admin_id, name])
for m in modules:
    conn.query("""
    insert into app.organization_requests_modules (request_id, billing_id, module_id)
    VALUES (%s, %s, %s)
    """, [request_id, billing_id, m])

conn.commit()
conn.close()

In [8]:
conn.close()

UPDATE ORGANIZATION DOMAINS

In [9]:
from src.server.models.user.UserHelper import UserHelper
from src import external_api
from src.runtime_constants import rr
from src.db.mysql import DbConnection

conn = DbConnection()
conn = DbConnection(rr.ENV_TEST, is_remote=True)

sheet_id = "1VpZ24bAZ84IJI-0nsX0VxcZOk0KB4qD_s8u95f8t3rw"
range = "Clients"
data = external_api.read_sheet(sheet_id, range, True)

for row in data:
    if row["Domain"] == "":
        continue
    domain = row["Domain"]
    is_admin = row["Admin"] == "1"
    email = row["Email"]

    email_id = UserHelper.get_or_create_email_id(conn=conn, email=email)

    conn.query("""
    UPDATE app.users_registration_requests r
       SET r.organization_domain = %s,
           r.organization_role = %s
     WHERE r.email = %s
    """, [domain, "admin" if is_admin else "member", email])

conn.commit()
conn.close()

CREATE ORGANIZATION FOR USER

In [10]:
from src.server.models.user.UserModel import UserModel
from src.server.models.organization.organization_model_v2 import OrganizationModelV2
from src.runtime_constants import rr
from src.db.mysql import DbConnection

conn = DbConnection()
conn = DbConnection(rr.ENV_TEST, is_remote=True)

input_data = [
    {
        "email": "jukka.hilvonen@nitrogames.fi",
        "domain": "nitrogames.fi",
        "role": "admin"
    }
]

for it in input_data:
    user_model = UserModel.load_by_email(conn=conn, email=it["email"])
    om = OrganizationModelV2.s__load_by_domain(conn=conn, email_domain=it["domain"])
    if om is None:
        om = OrganizationModelV2.s__create(
            conn=conn,
            dev_id_int=user_model.dev_id(),
            credits_value=25,
            email_domain=it["domain"],
            prime_number=user_model.get_prime_number(),
            users=[]
        )
    om.add_user(
        user_id=user_model.id(),
        role=it["role"]
    )

conn.commit()
conn.close()

ADD MODULES TO USER

In [1]:
from src.db.mysql import DbConnection
from src.server.models.user_constants import uc
from src.server.models.billing.billing_model import BillingModel
from src.server.models.billing.billing_data import BillingUtils
from src.server.models.user.UserModel import UserModel
from src.runtime_constants import rr

#conn = DbConnection()
conn = DbConnection(rr.ENV_PROD, is_remote=True)

email = "erik.hegely@xynthesis.io"
modules = [
    BillingUtils.MODULE_AUDITOR,
    BillingUtils.MODULE_TUNER,
    BillingUtils.MODULE_GAPS,
    BillingUtils.MODULE_GAMES_EXPLORER,
    BillingUtils.MODULE_PLAYERS_EXPLORER,
    BillingUtils.MODULE_STUDIES
]

admin_id = conn.select_one("""
SELECT u.id
  FROM app.users u
 WHERE u.guid = %s
""", [uc.ADMIN_USER_GUID])["id"]

row = conn.select_one_or_none("""
SELECT o.id as organization_id, u.id as user_id
  FROM app.users u,
       app.organization_users ou,
       app.organization o
 WHERE u.email = %s
   AND u.id = ou.user_id
   AND ou.organization_id = o.id
""", [email])

if row is None:
    # todo create org
    org_id = 0
    user_id = 0
else:
    org_id = row["organization_id"]
    user_id = row["user_id"]

billing_id = BillingModel.s__create(
    conn=conn,
    organization_id=org_id,
    user_id=admin_id,
    expire_months=12,
    payment_id=0,
    modules=[{"id": module_id, "seats": 1} for module_id in modules]
)

user_model = UserModel(conn=conn, user_id=user_id)

for module_id in modules:
    user_model.organization().add_licence_for_user(
        billing_id=billing_id,
        module_id=module_id
    )

#conn.rollback()
conn.commit()
conn.close()

REMOVE BILLING

In [6]:
from src.db.mysql import DbConnection
from src.server.models.billing.billing_model import BillingModel
from src.runtime_constants import rr

rr.set_env(rr.ENV_PROD)
#conn = DbConnection()
conn = DbConnection(rr.ENV_PROD, is_remote=True)

billing_id = 9

BillingModel.s__delete(
    conn=conn,
    billing_id=billing_id
)

conn.commit()
conn.close()

ADD USER REQUEST

In [1]:
from src.db.mysql import DbConnection
from src.runtime_constants import rr
from src.utils.gembase_utils import GembaseUtils

rr.set_env(rr.ENV_PROD)

name = "Pavel Dolezal"
email = "pavel.dolezal@gembase.io"
domain = None
role = "admin"
dev_id = 32190

#conn = DbConnection()
conn = DbConnection(rr.ENV_PROD, is_remote=True)

conn.query("""
        DELETE FROM app.users_registration_requests r
         WHERE r.email = %s
        """, [email])

guid = GembaseUtils.get_guid()
conn.query("""
INSERT INTO app.users_registration_requests (guid, email, name, dev_id_int, locked, credits, organization_domain, organization_role)
VALUES (%s, %s, %s, %s, 1, 25, %s, %s)
""", [guid, email, name, dev_id, domain, role])

conn.commit()
conn.close()

SCRAP DEV

In [2]:
from src.server.models.scraper_model import ScraperModel
from src.db.mysql import DbConnection
from src.runtime_constants import rr
from src.server.models.app_model import AppModel

rr.set_env(rr.ENV_PROD)
#conn = DbConnection()
conn = DbConnection(rr.ENV_PROD, is_remote=True)

dev_id_in_store = "7107438876846652382"
store = AppModel.STORE__GOOGLE_PLAY

res = ScraperModel.scrap_dev(
    conn=conn,
    dev_id=dev_id_in_store,
    store=store
)
print(f"dev_id_int: {res['dev_id_int']}")
for app_id_in_store in res["app_ids_in_store"]:
    ScraperModel.scrap_app(
        conn=conn,
        app_id_in_store=app_id_in_store,
        store=store
    )
conn.commit()
conn.close()

dev_id_int: 34451


CREATE CONCEPT DEV

In [1]:
from src.db.mysql import DbConnection
from src.runtime_constants import rr
from src.server.models.app_model import AppModel

#rr.set_env(rr.ENV_PROD)
conn = DbConnection()
#conn = DbConnection(rr.ENV_PROD, is_remote=True)

res = AppModel.create_dev_concept(
    conn=conn,
    title="Stillfront"
)
print(res["dev_id_int"])

conn.commit()
conn.close()

98680


ADD MODULES TO EXISTING USER WITHOUT COMPANY

In [2]:
from src.db.mysql import DbConnection
from src.server.models.user_constants import uc
from src.server.models.billing.billing_model import BillingModel
from src.server.models.billing.billing_data import BillingUtils
from src.server.models.user.UserModel import UserModel
from src.runtime_constants import rr
from src.server.models.organization.organization_model_v2 import OrganizationModelV2

email = "gustav.pastucha@gmail.com"
dev_id_int = 99944

rr.set_env(rr.ENV_PROD, remote=True)
conn = DbConnection()
#conn = DbConnection(rr.ENV_PROD, is_remote=True)

modules = [
    BillingUtils.MODULE_AUDITOR,
    BillingUtils.MODULE_TUNER,
    BillingUtils.MODULE_GAPS,
    BillingUtils.MODULE_GAMES_EXPLORER,
    BillingUtils.MODULE_PLAYERS_EXPLORER,
    BillingUtils.MODULE_STUDIES
]

admin_id = conn.select_one("""
SELECT u.id
  FROM app.users u
 WHERE u.guid = %s
""", [uc.ADMIN_USER_GUID])["id"]

user_model = UserModel.load_by_email(
    conn=conn,
    email=email
)

organization_model = OrganizationModelV2.s__load_by_user(
    conn=conn,
    user_id=user_model.id()
)

if organization_model is None:

    organization_model = OrganizationModelV2.s__create(
        dev_id_int=dev_id_int,
        email_domain=None,
        prime_number=user_model.prime_number(),
        credits_value=25,
        conn=conn,
        users=[]
    )
    organization_model.add_user(
        user_id=user_model.id(),
        role="member"
    )

billing_id = BillingModel.s__create(
    conn=conn,
    organization_id=organization_model.organization_id(),
    user_id=admin_id,
    expire_months=12,
    payment_id=0,
    modules=[{"id": module_id, "seats": 1} for module_id in modules]
)

for module_id in modules:
    organization_model.add_licence_for_user(
        user_id=user_model.id(),
        billing_id=billing_id,
        module_id=module_id
    )

#conn.rollback()
conn.commit()
conn.close()

CREATE USER REQUEST WITH MODULES

In [6]:
from src.server.models.billing.billing_data import BillingUtils
from src.runtime_constants import rr
from src.db.mysql import DbConnection
from src.server.models.user_constants import uc
from src.utils.gembase_utils import GembaseUtils
from src.server.models.organization.organization_model import OrganizationModel
from src.server.models.billing.billing_model import BillingModel
from src.server.models.app_model import AppModel
from src.server.models.user.UserModel import UserModel
from src.server.models.organization.organization_model_v2 import OrganizationModelV2
from src.runtime_constants import rr
from src.db.mysql import DbConnection
from src.server.models.user.UserHelper import UserHelper

#conn = DbConnection()
conn = DbConnection(rr.ENV_PROD, is_remote=True)

name = "Shum Singh"
email = "shum@agnitiocapital.com"
dev_id_int = 35590
domain = "agnitiocapital.com"
admin = True
modules = [
    BillingUtils.MODULE_AUDITOR,
    BillingUtils.MODULE_TUNER,
    BillingUtils.MODULE_GAPS,
    BillingUtils.MODULE_GAMES_EXPLORER,
    BillingUtils.MODULE_PLAYERS_EXPLORER,
    BillingUtils.MODULE_STUDIES
]

admin_id = conn.select_one("""
SELECT u.id
  FROM app.users u
 WHERE u.guid = %s
""", [uc.ADMIN_USER_GUID])["id"]
if dev_id_int is None:
    dev_data = AppModel.create_dev_concept(
        conn=conn,
        title=company
    )
    dev_id_int = dev_data["dev_id_int"]
prime_number = conn.select_one("""
SELECT p.id
  FROM app.def_prime_numbers p
  ORDER BY RAND()
  LIMIT 1
""")["id"]
org = OrganizationModelV2.s__load_by_domain(conn=conn, email_domain=email)
if org is None:
    org = OrganizationModelV2.s__create(conn=conn, dev_id_int=dev_id_int, prime_number=prime_number,
                                  users=[], email_domain=domain, credits_value=25)
    org_id = org.organization_id()
else:
    org_id = org.organization_id()

modules_seats = [{"id": m, "seats": 1} for m in modules]
billing_id = BillingModel.s__create(
    conn=conn,
    organization_id=org_id,
    user_id=0,
    payment_id=0,
    expire_months=12,
    modules=modules_seats
)
email_id = UserHelper.get_or_create_email_id(conn=conn, email=email)
conn.query("""
DELETE FROM app.organization_requests WHERE email = %s
""", [email])
request_id = conn.insert(f"""
insert into app.organization_requests (organization_id, email, request_guid, from_user_id, role, name, locked)
VALUES (%s, %s, %s, %s, {'admin' if admin else 'member'}, %s, 1)
""", [org_id, email, GembaseUtils.get_guid(), admin_id, name])
for m in modules:
    conn.query("""
    insert into app.organization_requests_modules (request_id, billing_id, module_id)
    VALUES (%s, %s, %s)
    """, [request_id, billing_id, m])

conn.commit()
conn.close()

In [5]:
conn.close()